# Embeddings 1.0 + checks

In [2]:
import pandas as pd
df = pd.read_csv("data/processed/movies_cleaned.csv")

print(df.shape)
df.head()

(9967, 12)


,adult,id,original_language,original_title,overview,popularity,release_date,title,vote_average,vote_count,movie_id,embedding_text
0,False,278,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,46.3708,1994-09-23,The Shawshank Redemption,8.718,30171,278,Imprisoned in the 1940s for the double murder ...
1,False,238,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",42.0006,1972-03-14,The Godfather,8.686,22787,238,"Spanning the years 1945 to 1955, a chronicle o..."
2,False,240,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,26.8671,1974-12-20,The Godfather Part II,8.571,13812,240,In the continuing saga of the Corleone crime f...
3,False,424,en,Schindler's List,The true story of how businessman Oskar Schind...,24.2944,1993-12-15,Schindler's List,8.567,17341,424,The true story of how businessman Oskar Schind...
4,False,389,en,12 Angry Men,The defense and the prosecution have rested an...,19.4971,1957-04-10,12 Angry Men,8.559,9908,389,The defense and the prosecution have rested an...


In [4]:
from transformers import AutoTokenizer
import pandas as pd

c:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
models = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "mpnet": "sentence-transformers/all-mpnet-base-v2",
    "gte_modernbert": "Alibaba-NLP/gte-modernbert-base",
    #"embeddinggemma": "google/embeddinggemma-300m", kasnije
}

In [6]:
token_length_results = []

texts = df["embedding_text"].tolist()

for model_name, model_id in models.items():
    print(f"Obrada: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    token_lengths = [
        len(
            tokenizer(
                text,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )
        for text in texts
    ]

    token_length_results.append(
        {
            "model": model_name,
            "min_tokens": min(token_lengths),
            "mean_tokens": sum(token_lengths) / len(token_lengths),
            "median_tokens": pd.Series(token_lengths).median(),
            "p90_tokens": pd.Series(token_lengths).quantile(0.90),
            "p95_tokens": pd.Series(token_lengths).quantile(0.95),
            "p99_tokens": pd.Series(token_lengths).quantile(0.99),
            "max_tokens": max(token_lengths),
            "over_256_count": sum(length > 256 for length in token_lengths),
            "over_256_percentage": (
                sum(length > 256 for length in token_lengths)
                / len(token_lengths)
                * 100
            ),
        }
    )

Obrada: minilm


c:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikola.bakic\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Obrada: mpnet


c:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikola.bakic\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Obrada: gte_modernbert


c:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikola.bakic\.cache\huggingface\hub\models--Alibaba-NLP--gte-modernbert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [7]:
token_length_summary = pd.DataFrame(token_length_results)

display(token_length_summary)

,model,min_tokens,mean_tokens,median_tokens,p90_tokens,p95_tokens,p99_tokens,max_tokens,over_256_count,over_256_percentage
0,minilm,6,58.162837,52.0,96.0,108.0,145.00,224,0,0.0
1,mpnet,6,58.162837,52.0,96.0,108.0,145.00,224,0,0.0
2,gte_modernbert,6,59.296478,53.0,97.0,110.0,146.34,235,0,0.0


In [8]:
import gc

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

In [9]:
model_registry = {
    "minilm": {
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "expected_dimension": 384,
    },
    "mpnet": {
        "model_id": "sentence-transformers/all-mpnet-base-v2",
        "expected_dimension": 768,
    },
    "gte_modernbert": {
        "model_id": "Alibaba-NLP/gte-modernbert-base",
        "expected_dimension": 768,
    },
}

smoke_test_texts = df["embedding_text"].head(8).tolist()
smoke_test_results = []

for model_name, config in model_registry.items():
    print(f"Testiranje modela: {model_name}")

    model = SentenceTransformer(
        config["model_id"],
        device="cpu",
    )
    model.max_seq_length = 256

    embeddings = model.encode(
        smoke_test_texts,
        batch_size=8,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    smoke_test_results.append(
        {
            "model": model_name,
            "shape": embeddings.shape,
            "expected_dimension": config["expected_dimension"],
            "contains_nan": bool(np.isnan(embeddings).any()),
            "contains_inf": bool(np.isinf(embeddings).any()),
            "mean_vector_norm": float(
                np.linalg.norm(embeddings, axis=1).mean()
            ),
            "passed": (
                embeddings.shape
                == (len(smoke_test_texts), config["expected_dimension"])
                and not np.isnan(embeddings).any()
                and not np.isinf(embeddings).any()
            ),
        }
    )

    del model, embeddings
    gc.collect()

Testiranje modela: minilm


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5638.90it/s]


Testiranje modela: mpnet


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4557.26it/s]


Testiranje modela: gte_modernbert


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 4570.56it/s]


In [10]:
smoke_test_summary = pd.DataFrame(smoke_test_results)
display(smoke_test_summary)

,model,shape,expected_dimension,contains_nan,contains_inf,mean_vector_norm,passed
0,minilm,"(8, 384)",384,False,False,1.0,True
1,mpnet,"(8, 768)",768,False,False,1.0,True
2,gte_modernbert,"(8, 768)",768,False,False,1.0,True


In [11]:
from pathlib import Path
from huggingface_hub.constants import HF_HUB_CACHE

for path in Path(HF_HUB_CACHE).glob("models--*"):
    print(path.name)

models--Alibaba-NLP--gte-modernbert-base
models--sentence-transformers--all-MiniLM-L6-v2
models--sentence-transformers--all-mpnet-base-v2
